# Revenue Analysis — Take-Home Exercise

**Deliverables:** total revenue, revenue by segment, top 5 customers by revenue, net revenue after promotional discounts.

**How I approached this.** I audited each file before joining anything, wrote down every place the data forced a
judgment call, picked a defensible default, and then re-ran the whole thing under the alternative assumptions so the
sensitivity of each number is visible rather than hidden. Where a choice materially moves a number, I say so and give
the number both ways.

**Headline decisions** (each justified in its own section below):

| # | Issue found | Decision |
|---|---|---|
| 1 | 300 exact duplicate order rows | Drop — same `order_id`, byte-identical rows |
| 2 | 90 orders whose `customer_id` isn't in `customers.csv` | Keep in company totals, bucket as `Unknown` in segment view, exclude from top-customer ranking |
| 3 | Three order statuses (`completed`, `refunded`, `cancelled`) | Revenue = `completed` only; refunded/cancelled reported separately |
| 4 | 383 orders carry two promotions | Aggregate promotions to one row per order *before* joining, to avoid join fan-out |
| 5 | 73 orders where total discount exceeds the order amount | Cap discount at the order amount; also report uncapped |
| 6 | **`Mid-Market` has 612 customers and zero orders** | Report as an explicit `0` row and flag as a pipeline gap — do not impute |
| 7 | Unknown customer IDs form a separate `199xxx` namespace | Treated as a second ID scheme; flagged as the key open question |

## 0. Setup and load

In [1]:
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
SRC = "./"

# Load IDs as strings: they are identifiers, not quantities. Reading them as int64 risks
# silent coercion to float (and precision loss) if any file ever ships a blank ID.
customers_raw = pd.read_csv(SRC + "customers.csv", dtype={"customer_id": str})
orders_raw    = pd.read_csv(SRC + "orders.csv",    dtype={"order_id": str, "customer_id": str})
promos_raw    = pd.read_csv(SRC + "promotions.csv", dtype={"promotion_id": str, "order_id": str})

for name, df in [("customers", customers_raw), ("orders", orders_raw), ("promotions", promos_raw)]:
    print(f"{name:<12} {df.shape[0]:>6,} rows x {df.shape[1]} cols   |   {list(df.columns)}")

customers     3,000 rows x 4 cols   |   ['customer_id', 'customer_name', 'segment', 'region']
orders       30,390 rows x 5 cols   |   ['order_id', 'customer_id', 'order_date', 'amount', 'status']
promotions    2,812 rows x 4 cols   |   ['promotion_id', 'order_id', 'promo_code', 'discount_amount']


## 1. Data audit

Everything below is run *before* any cleaning, so the decisions that follow are grounded in observed counts.

In [2]:
for name, df in [("customers", customers_raw), ("orders", orders_raw), ("promotions", promos_raw)]:
    print(f"===== {name} =====")
    print("nulls per column:", df.isna().sum().to_dict())
    print("fully duplicated rows:", int(df.duplicated().sum()))
    print()

===== customers =====
nulls per column: {'customer_id': 0, 'customer_name': 0, 'segment': 0, 'region': 0}
fully duplicated rows: 0

===== orders =====
nulls per column: {'order_id': 0, 'customer_id': 0, 'order_date': 0, 'amount': 0, 'status': 0}
fully duplicated rows: 300

===== promotions =====
nulls per column: {'promotion_id': 0, 'order_id': 0, 'promo_code': 0, 'discount_amount': 0}
fully duplicated rows: 0



In [3]:
# Categorical columns: check for the usual mess (casing variants, stray whitespace, synonyms).
print("segment:", customers_raw.segment.value_counts(dropna=False).to_dict())
print("region: ", customers_raw.region.value_counts(dropna=False).to_dict())
print("status: ", orders_raw.status.value_counts(dropna=False).to_dict())
print("promo:  ", promos_raw.promo_code.value_counts(dropna=False).to_dict())

# Stray leading/trailing whitespace anywhere in the string columns?
for name, df in [("customers", customers_raw), ("orders", orders_raw), ("promotions", promos_raw)]:
    for col in df.columns:
        s = df[col].dropna().astype(str)
        n = int((s != s.str.strip()).sum())
        if n:
            print(f"WHITESPACE {name}.{col}: {n}")
print("\nCategoricals are clean: no casing variants, synonyms, or padding to normalise.")

segment: {'SMB': 1665, 'Enterprise': 723, 'Mid-Market': 612}
region:  {'Midwest': 758, 'South': 756, 'Northeast': 755, 'West': 731}
status:  {'completed': 25832, 'refunded': 2415, 'cancelled': 2143}
promo:   {'WELCOME15': 425, 'REFERRAL5': 415, 'BUNDLE20': 408, 'SPRING10': 406, 'FLASH25': 402, 'VIP30': 399, 'LOYALTY10': 357}

Categoricals are clean: no casing variants, synonyms, or padding to normalise.


In [4]:
# Numeric and date fields: do they parse, and are the values plausible?
amount = pd.to_numeric(orders_raw.amount, errors="coerce")
disc   = pd.to_numeric(promos_raw.discount_amount, errors="coerce")
dates  = pd.to_datetime(orders_raw.order_date, errors="coerce", format="%Y-%m-%d")

print(f"amount   -> unparseable {amount.isna().sum()}, negative {(amount < 0).sum()}, zero {(amount == 0).sum()}")
print(f"           min {amount.min():,.2f}  median {amount.median():,.2f}  max {amount.max():,.2f}")
print(f"discount -> unparseable {disc.isna().sum()}, negative {(disc < 0).sum()}")
print(f"           min {disc.min():,.2f}  median {disc.median():,.2f}  max {disc.max():,.2f}")
print(f"dates    -> unparseable {dates.isna().sum()},  range {dates.min().date()} to {dates.max().date()}")

amount   -> unparseable 0, negative 0, zero 0
           min 32.40  median 488.15  max 8,265.48
discount -> unparseable 0, negative 0
           min 10.01  median 97.13  max 199.94
dates    -> unparseable 0,  range 2025-07-01 to 2026-06-29


No parsing problems, no nulls, no negative or zero amounts, and no impossible dates — the dates span a clean
2025-07-01 to 2026-06-29 window with no future-dated orders. **So the messiness here is not dirty values; it is
relational.** The next four checks are where the real problems are.

In [5]:
# (a) Duplicate order rows
dup_ids = orders_raw.order_id[orders_raw.order_id.duplicated()].unique()
dup_rows = orders_raw[orders_raw.order_id.isin(dup_ids)]
conflicting = dup_rows.groupby("order_id").nunique().gt(1).any(axis=1).sum()
print(f"(a) duplicated order_ids: {len(dup_ids)}   groups with conflicting field values: {conflicting}")

# (b) Referential integrity
orphans = orders_raw[~orders_raw.customer_id.isin(customers_raw.customer_id)]
print(f"(b) orders with an unknown customer_id: {len(orphans)}  "
      f"across {orphans.customer_id.nunique()} missing customer_ids")
print(f"    promotions pointing at a missing order_id: "
      f"{(~promos_raw.order_id.isin(orders_raw.order_id)).sum()}")

# (c) Promotion cardinality — is order_id unique in promotions?
per_order = promos_raw.order_id.value_counts()
print(f"(c) promotions per order: {per_order.value_counts().sort_index().to_dict()}")

# (d) Does any promoted order sit on a non-completed status?
promo_status = promos_raw.merge(orders_raw.drop_duplicates("order_id"), on="order_id").status.value_counts()
print(f"(d) status of promoted orders: {promo_status.to_dict()}")

# (e) What do the unknown customer_ids look like? Random, or a structured block?
print(f"(e) known customer_id range   : {customers_raw.customer_id.astype(int).min()} "
      f"- {customers_raw.customer_id.astype(int).max()}")
print(f"    unknown customer_id range : {orphans.customer_id.astype(int).min()} "
      f"- {orphans.customer_id.astype(int).max()}")
print(f"    all unknown IDs >= 199000 : {(orphans.customer_id.astype(int) >= 199_000).all()}")

# (f) Order coverage per segment — does every segment actually transact?
coverage = (customers_raw
            .merge(orders_raw.drop_duplicates().groupby("customer_id").size().rename("n_orders"),
                   left_on="customer_id", right_index=True, how="left")
            .assign(n_orders=lambda d: d.n_orders.fillna(0))
            .groupby("segment")
            .agg(customers=("customer_id", "size"),
                 customers_with_orders=("n_orders", lambda s: int((s > 0).sum())),
                 total_orders=("n_orders", "sum")))
print("\n(f) order coverage by segment:")
print(coverage.to_string())

(a) duplicated order_ids: 300   groups with conflicting field values: 0
(b) orders with an unknown customer_id: 90  across 43 missing customer_ids
    promotions pointing at a missing order_id: 0
(c) promotions per order: {1: 2046, 2: 383}
(d) status of promoted orders: {'completed': 2812}
(e) known customer_id range   : 100001 - 103000
    unknown customer_id range : 199001 - 199050
    all unknown IDs >= 199000 : True

(f) order coverage by segment:
            customers  customers_with_orders  total_orders
segment                                                   
Enterprise        723                    723      9,093.00
Mid-Market        612                      0          0.00
SMB              1665                   1665     20,907.00


### Two structural findings, not value errors

Checks (e) and (f) turn up the two most consequential things in this dataset, and neither is a dirty value:

**(e) The unknown customer IDs are a distinct namespace.** All 43 of them sit in a `199001`-`199050` block, entirely
disjoint from the `100001`-`103000` range the customers file uses. A random sync failure would scatter missing IDs
through the known range; a contiguous block in a different range means these orders came from a **different system or
ID scheme** — a second instance, an acquisition, or a test/staging block that leaked into the extract. Their order
amounts are statistically indistinguishable from the matched orders, so they behave like genuine transactions.

**(f) `Mid-Market` has zero orders — all 612 of them.** Every single Enterprise customer (723/723) and every single SMB
customer (1,665/1,665) has at least one order. Not one of the 612 Mid-Market customers does. Perfect coverage in two
segments and perfect absence in the third is not customer behaviour; it is a **pipeline gap**. Whatever produced
`orders.csv` did not include Mid-Market activity.

This is the finding I would lead with in a real report, because it is the one that would cause someone to draw a
badly wrong conclusion. "Mid-Market generated no revenue" is a story about a broken extract, not about the segment.
Handling in §2.6.

### The trap this sets up

`order_id` is **not** unique in `orders` (300 dupes) and **not** unique in `promotions` (383 orders carry two promo
codes). A naive `orders.merge(promotions, on="order_id")` therefore fans out and inflates revenue. Quantifying it so
the fix is not taken on faith:

In [6]:
naive = orders_raw.merge(promos_raw, on="order_id", how="left")
naive_rev = pd.to_numeric(naive.loc[naive.status == "completed", "amount"]).sum()
clean_rev = pd.to_numeric(
    orders_raw.drop_duplicates().query("status == 'completed'").amount
).sum()
print(f"rows after naive left join : {len(naive):,}  (from {len(orders_raw):,} orders)")
print(f"revenue if computed there  : {naive_rev:>15,.2f}")
print(f"revenue done correctly     : {clean_rev:>15,.2f}")
print(f"overstatement avoided      : {naive_rev - clean_rev:>15,.2f}  "
      f"({(naive_rev / clean_rev - 1):.2%})")

rows after naive left join : 30,775  (from 30,390 orders)
revenue if computed there  :   16,553,978.28
revenue done correctly     :   16,153,512.87
overstatement avoided      :      400,465.41  (2.48%)


## 2. Cleaning decisions

### 2.1 Duplicate orders — drop them

300 `order_id`s appear twice, and every duplicate group is byte-identical across all five columns (0 conflicting
groups above). Two genuinely distinct orders would need distinct `order_id`s, so identical rows sharing one ID are an
ingestion artefact — a re-run or double append — not real repeat business.

I drop exact duplicates rather than deduplicating on `order_id` alone. Both give the same result here, but the
stricter form would *fail loudly* if a future load contained the same ID with a different amount, which is a genuine
data-quality incident that shouldn't be silently resolved by `keep="first"`.

### 2.2 Orders with an unknown customer — keep in the total, bucket as `Unknown`

90 orders (43 customer IDs) reference customers absent from `customers.csv`. Given finding (e) — the IDs form a
contiguous `199xxx` block outside the customers file's namespace — my read is a second ID scheme rather than a
snapshot-timing issue. The orders themselves look entirely normal, with an amount distribution matching the matched
orders.

I flag it as the open question it is, because the two candidate explanations point opposite ways: if these are real
accounts from another system, the money belongs in the total; if the `199xxx` block is test or staging data, all 90
orders should be dropped. I cannot tell from this extract, so I include them (a $47K, 0.29% effect) and make the
inclusion visible and reversible rather than burying it.

Deleting them would understate company revenue for a reason that has nothing to do with the orders. So:

- **Total and net revenue: included.** The money was real; the dimension table is incomplete.
- **Segment breakdown: shown as its own `Unknown` row** rather than dropped or guessed. This keeps the segment column
  summing to the total, which means a reader can immediately see the size of the gap instead of wondering why the
  parts don't add up.
- **Top-5 customers: excluded.** Ranking rows with no name would be meaningless, and their individual amounts are far
  too small to reach the top 5 — verified below rather than assumed.

### 2.3 Status — revenue is `completed` only

Three statuses: `completed`, `refunded`, `cancelled`. `cancelled` is uncontroversial: no transaction occurred.

`refunded` is the real judgment call. Under accrual accounting a refunded order was recognised as revenue and then
reversed, so its net contribution is zero — the same as excluding it. Under a "gross bookings" view it would count.
I exclude it, because "revenue" without qualification normally means recognised, retained revenue, and counting money
that was handed back would overstate the business by ~9%.

I report all three buckets explicitly so a stakeholder who wants gross bookings can read it off directly, and I
quantify the swing in the sensitivity table in §4.

### 2.4 Promotions — aggregate to order level before joining

383 orders carry two promo codes (stacked discounts). Rather than join and then deduplicate, I collapse promotions to
one row per order first — sum the discounts, keep the codes for reference — which makes the join provably one-to-one
and the fan-out structurally impossible instead of merely corrected after the fact.

Summing stacked discounts is the right reading: two distinct `promotion_id`s with different codes on one order are two
concessions granted, not one recorded twice. (If they were duplicates I'd expect the *same* code repeated, which is
not the pattern present.)

Usefully, every promoted order is `completed` (check (d) above), so there's no awkward question about how to treat a
discount attached to an order that was cancelled.

### 2.6 `Mid-Market` — report it as an explicit zero, never let it be a missing row

The dangerous handling here is to do nothing. A `groupby("segment")` on the orders simply won't emit a Mid-Market row,
and a reader scanning the table has no way to distinguish "this segment isn't in the data" from "I forgot to look."

So I reindex the segment table against the full segment list from `customers.csv`, forcing Mid-Market to appear as an
explicit `0` with its customer count attached. A zero next to "612 customers, 0 with orders" is self-evidently a data
problem; an absent row is invisible.

I do **not** impute or estimate Mid-Market revenue. With no transactional signal at all, any number would be fabricated,
and 612 of 3,000 customers is too large a share to paper over.

### 2.5 Discounts exceeding the order amount — cap at the order amount

60 individual promotions, and 73 orders once stacking is accounted for, carry a discount larger than the order itself.
Taken literally these produce negative net revenue: the company paid the customer to buy something.

That is not plausible as a business fact, so it's a data error — most likely discounts recorded as list-price
reductions against a since-changed amount, or a percentage stored as an absolute. Without a source system to check, I
cap discount at the order amount, flooring per-order net revenue at zero. This is the conservative choice in the sense
that matters: it avoids fabricating negative revenue that no accounting system would ever report.

I compute the uncapped figure too. The difference is small enough that it doesn't change any conclusion, which is
itself worth stating.

In [7]:
# --- 2.1 dedupe orders -------------------------------------------------------
orders = orders_raw.drop_duplicates().copy()
assert orders.order_id.is_unique, "order_id still not unique after dropping exact duplicates"
print(f"orders: {len(orders_raw):,} -> {len(orders):,}  ({len(orders_raw) - len(orders)} exact duplicates dropped)")

# --- typing ------------------------------------------------------------------
orders["amount"] = pd.to_numeric(orders.amount)
orders["order_date"] = pd.to_datetime(orders.order_date, format="%Y-%m-%d")

# --- 2.4 collapse promotions to one row per order ----------------------------
promos = (promos_raw
          .assign(discount_amount=lambda d: pd.to_numeric(d.discount_amount))
          .groupby("order_id")
          .agg(discount_raw=("discount_amount", "sum"),
               promo_codes=("promo_code", lambda s: " + ".join(sorted(s))),
               n_promos=("promotion_id", "size"))
          .reset_index())
assert promos.order_id.is_unique
print(f"promotions: {len(promos_raw):,} rows -> {len(promos):,} order-level rows "
      f"({(promos.n_promos > 1).sum()} orders with stacked promos)")

# --- join (now provably one-to-one) ------------------------------------------
df = (orders
      .merge(promos, on="order_id", how="left", validate="one_to_one")
      .merge(customers_raw, on="customer_id", how="left", validate="many_to_one"))
df["discount_raw"] = df.discount_raw.fillna(0.0)
assert len(df) == len(orders), "join changed the row count"

# --- 2.5 cap discount at the order amount ------------------------------------
df["discount"] = df[["discount_raw", "amount"]].min(axis=1)
capped = df.discount_raw > df.amount
print(f"discount capped on {capped.sum()} orders "
      f"(reduced total discount by {(df.discount_raw - df.discount).sum():,.2f})")

# --- 2.2 label unmatched customers -------------------------------------------
df["is_known_customer"] = df.customer_name.notna()
df["segment"] = df.segment.fillna("Unknown")
print(f"orders with no customer record: {(~df.is_known_customer).sum()}")
print(f"\nanalysis table: {len(df):,} rows x {df.shape[1]} cols")

orders: 30,390 -> 30,090  (300 exact duplicates dropped)
promotions: 2,812 rows -> 2,429 order-level rows (383 orders with stacked promos)
discount capped on 73 orders (reduced total discount by 2,988.59)
orders with no customer record: 90

analysis table: 30,090 rows x 13 cols


In [8]:
# Sanity check on the §2.2 claim that unmatched orders can't reach the top 5.
unknown_by_id = (df[~df.is_known_customer & (df.status == "completed")]
                 .groupby("customer_id").amount.sum())
known_by_id = (df[df.is_known_customer & (df.status == "completed")]
               .groupby("customer_id").amount.sum())
print(f"largest unmatched customer_id by revenue: {unknown_by_id.max():,.2f}")
print(f"5th-largest known customer by revenue:    {known_by_id.nlargest(5).iloc[-1]:,.2f}")
print("=> excluding unmatched customers cannot change the top-5 ranking.")

largest unmatched customer_id by revenue: 7,326.23
5th-largest known customer by revenue:    16,448.81
=> excluding unmatched customers cannot change the top-5 ranking.


## 3. Results

In [9]:
completed = df[df.status == "completed"]

total_revenue = completed.amount.sum()
total_discount = completed.discount.sum()
net_revenue = total_revenue - total_discount

print("1. TOTAL REVENUE (completed orders)")
print(f"   {total_revenue:>18,.2f}   across {len(completed):,} orders\n")
print("   by status, for reference:")
print(df.groupby("status").amount.agg(orders="size", value="sum").to_string(), "\n")
print("4. NET REVENUE (after promotional discounts)")
print(f"   gross            {total_revenue:>18,.2f}")
print(f"   discounts        {-total_discount:>18,.2f}   "
      f"({len(completed[completed.discount > 0]):,} promoted orders)")
print(f"   net              {net_revenue:>18,.2f}   "
      f"(discount rate {total_discount / total_revenue:.2%})")

1. TOTAL REVENUE (completed orders)
        16,153,512.87   across 25,575 orders

   by status, for reference:
           orders         value
status                         
cancelled    2119  1,321,445.08
completed   25575 16,153,512.87
refunded     2396  1,501,574.05 

4. NET REVENUE (after promotional discounts)
   gross                 16,153,512.87
   discounts               -273,837.47   (2,429 promoted orders)
   net                   15,879,675.40   (discount rate 1.70%)


In [10]:
print("2. REVENUE BY CUSTOMER SEGMENT\n")
# Reindex against every segment in customers.csv so a segment with no orders shows a
# visible zero instead of silently vanishing from the report (see 2.6).
all_segments = sorted(set(customers_raw.segment) | {"Unknown"})

by_segment = (completed
              .groupby("segment")
              .agg(orders=("order_id", "size"),
                   revenue=("amount", "sum"),
                   discounts=("discount", "sum"))
              .reindex(all_segments)
              .fillna(0)
              .assign(net_revenue=lambda d: d.revenue - d.discounts,
                      pct_of_revenue=lambda d: d.revenue / d.revenue.sum() * 100,
                      avg_order_value=lambda d: (d.revenue / d.orders).fillna(0))
              .astype({"orders": int})
              .sort_values("revenue", ascending=False))

print(by_segment.to_string())
print("\n   !! Mid-Market shows 0 because orders.csv contains no Mid-Market activity at all")
print("      (612 customers, 0 with orders) - a pipeline gap, NOT zero demand. See 2.6.")
print(f"\n   checksum: segment revenue sums to {by_segment.revenue.sum():,.2f} "
      f"= total revenue {total_revenue:,.2f} -> {by_segment.revenue.sum() == total_revenue}")

2. REVENUE BY CUSTOMER SEGMENT

            orders       revenue  discounts   net_revenue  pct_of_revenue  avg_order_value
segment                                                                                   
SMB          17772 11,211,218.26 192,124.91 11,019,093.35           69.40           630.84
Enterprise    7729  4,895,261.50  80,670.69  4,814,590.81           30.30           633.36
Unknown         74     47,033.11   1,041.87     45,991.24            0.29           635.58
Mid-Market       0          0.00       0.00          0.00            0.00             0.00

   !! Mid-Market shows 0 because orders.csv contains no Mid-Market activity at all
      (612 customers, 0 with orders) - a pipeline gap, NOT zero demand. See 2.6.

   checksum: segment revenue sums to 16,153,512.87 = total revenue 16,153,512.87 -> True


In [11]:
print("3. TOP 5 CUSTOMERS BY REVENUE (known customers only)\n")
top5 = (completed[completed.is_known_customer]
        .groupby(["customer_id", "customer_name", "segment", "region"])
        .agg(orders=("order_id", "size"),
             revenue=("amount", "sum"),
             discounts=("discount", "sum"))
        .assign(net_revenue=lambda d: d.revenue - d.discounts)
        .sort_values("revenue", ascending=False)
        .head(5)
        .reset_index())
top5.index = range(1, len(top5) + 1)
print(top5.to_string())
print(f"\n   top 5 = {top5.revenue.sum() / total_revenue:.2%} of total revenue")
ranked = (completed[completed.is_known_customer].groupby("customer_id").amount.sum()
          .sort_values(ascending=False))
gap = ranked.iloc[4] - ranked.iloc[5]
print(f"   #5 = {ranked.iloc[4]:,.2f} vs #6 = {ranked.iloc[5]:,.2f}  ->  gap of {gap:,.2f} ({gap / ranked.iloc[4]:.2%})")
print("   Note: revenue is thinly spread (3,000 customers, no whales), so the top-5 cut is")
print("   close-run — the gap between #5 and #6 is small and would move under a different")
print("   status rule. Reported as-is, but I wouldn't build an account strategy on the exact order.")

3. TOP 5 CUSTOMERS BY REVENUE (known customers only)

  customer_id       customer_name     segment     region  orders   revenue  discounts  net_revenue
1      102421     Summit Ventures  Enterprise  Northeast      22 18,192.94       0.00    18,192.94
2      100336   Solstice Labs LLC         SMB       West      23 17,747.97     134.73    17,613.24
3      101221    Amber Consulting         SMB      South      17 17,276.74     293.80    16,982.94
4      102634        Cobalt Group         SMB  Northeast      21 16,582.14       0.00    16,582.14
5      100724  Lakeside Foods LLC         SMB      South      16 16,448.81     155.15    16,293.66

   top 5 = 0.53% of total revenue
   #5 = 16,448.81 vs #6 = 16,209.96  ->  gap of 238.85 (1.45%)
   Note: revenue is thinly spread (3,000 customers, no whales), so the top-5 cut is
   close-run — the gap between #5 and #6 is small and would move under a different
   status rule. Reported as-is, but I wouldn't build an account strategy on the exact o

## 4. Sensitivity to the judgment calls

The point of this table is that none of the §2 decisions are load-bearing in secret. If the reviewer's assumptions
differ from mine, the number they'd get is here.

In [12]:
gross_bookings = df[df.status.isin(["completed", "refunded"])].amount.sum()

scenarios = pd.DataFrame([
    ("Baseline (completed only, discounts capped)", total_revenue, net_revenue),
    ("Include refunded as revenue (gross bookings)", gross_bookings,
     gross_bookings - df[df.status.isin(["completed", "refunded"])].discount.sum()),
    ("Drop orders with unknown customer", 
     completed[completed.is_known_customer].amount.sum(),
     completed[completed.is_known_customer].eval("amount - discount").sum()),
    ("Discounts uncapped (allow negative net)", total_revenue,
     total_revenue - completed.discount_raw.sum()),
    ("Duplicate orders NOT dropped", 
     orders_raw.query("status == 'completed'").amount.astype(float).sum(), float("nan")),
], columns=["scenario", "total_revenue", "net_revenue"])

scenarios["delta_vs_baseline"] = scenarios.total_revenue - total_revenue
scenarios["delta_pct"] = scenarios.delta_vs_baseline / total_revenue * 100
print(scenarios.to_string(index=False))

                                    scenario  total_revenue   net_revenue  delta_vs_baseline  delta_pct
 Baseline (completed only, discounts capped)  16,153,512.87 15,879,675.40               0.00       0.00
Include refunded as revenue (gross bookings)  17,655,086.92 17,381,249.45       1,501,574.05       9.30
           Drop orders with unknown customer  16,106,479.76 15,833,684.16         -47,033.11      -0.29
     Discounts uncapped (allow negative net)  16,153,512.87 15,876,686.81               0.00       0.00
                Duplicate orders NOT dropped  16,314,305.50           NaN         160,792.63       1.00


Reading it: only the refund treatment moves the headline materially (≈+9%). The orphan-customer and discount-cap
decisions are sub-1% effects, so they're worth documenting for correctness but they don't change any conclusion.
Failing to drop the duplicate orders would have overstated revenue by ~1% — invisible in a summary figure, which is
exactly why it's worth an assertion in code rather than a spot check.

## 5. What I'd ask before this went into a report

1. **Refund semantics.** Is `refunded` a full or partial refund, and is the refund dated to the original order? Both
   affect period-over-period revenue, and neither is answerable from this extract.
2. **Where is Mid-Market?** The single most important question here. 612 customers, zero orders, while the other two
   segments have 100% coverage. Is Mid-Market activity in a separate system, filtered out of this extract, or booked
   under a different segment label? Until this is answered, the segment breakdown is incomplete and no
   segment-comparison conclusion should be published from it.
3. **The `199xxx` customer_id block.** Real accounts from another instance, or test data? This determines whether 90
   orders and ~$47K belong in the total at all. The contiguous ID range makes me want to see the source system before
   trusting them.
4. **Discounts above order value.** I capped them, but the source system would say whether `amount` is pre- or
   post-discount. If it is already *post*-discount, then subtracting discounts double-counts them and net revenue is
   simply the total — a materially different answer that no amount of cleaning can settle from the data alone.
5. **Stacked promo policy.** Is stacking two codes intended? 383 orders is too many to be accidental, but worth
   confirming rather than assuming.